# Hottest & Coldest Stocks (Change Between Filings)
This notebook loads the raw 13F data and calculates the largest aggregate increases (Hottest) and largest aggregate drops (Coldest) in position values across tracked hedge funds, between the two most recent reporting periods.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load dataset
df = pd.read_csv('../data_by_stocks.zip', low_memory=False)
df['reportDate'] = pd.to_datetime(df['reportDate'])
df.head()

In [ ]:
# Identify the two most recent reporting periods
unique_periods = sorted(df['reportDate'].unique(), reverse=True)
if len(unique_periods) < 2:
    raise ValueError("Not enough periods to compare changes.")

current_period = unique_periods[0]
previous_period = unique_periods[1]

print(f"Current Period: {pd.to_datetime(current_period).strftime('%Y-%m-%d')}")
print(f"Previous Period: {pd.to_datetime(previous_period).strftime('%Y-%m-%d')}")

# Filter data to just these two periods
df_current = df[df['reportDate'] == current_period]
df_previous = df[df['reportDate'] == previous_period]

In [ ]:
# Aggregate values by Issuer for both periods
agg_current = df_current.groupby('nameOfIssuer')['value'].sum()
agg_previous = df_previous.groupby('nameOfIssuer')['value'].sum()

# Combine into a single DataFrame
change_df = pd.DataFrame({
    'current_value': agg_current,
    'previous_value': agg_previous
}).fillna(0) # Fill NaNs with 0 for stocks not held in one of the periods

# Calculate the difference and convert to Millions ($ MMs)
change_df['change_value_MMs'] = ((change_df['current_value'] - change_df['previous_value']) / 1000).round(1)

# Separate into Hottest (largest increase) and Coldest (largest drop)
hottest = change_df.sort_values(by='change_value_MMs', ascending=False).head(20).reset_index()
coldest = change_df.sort_values(by='change_value_MMs', ascending=True).head(20).reset_index()


In [ ]:
display(hottest[['nameOfIssuer', 'change_value_MMs']].head(5))
display(coldest[['nameOfIssuer', 'change_value_MMs']].head(5))

In [ ]:
# Visualize Hottest Stocks
plt.figure(figsize=(12, 8))
barplot_hot = sns.barplot(x='change_value_MMs', y='nameOfIssuer', data=hottest, palette='Greens_r')

plt.title(f'Hottest Stocks: Largest Increase ({pd.to_datetime(previous_period).strftime("%b %Y")} to {pd.to_datetime(current_period).strftime("%b %Y")})', fontsize=16, pad=15)
plt.xlabel('Change in Aggregated Value ($ MMs)', fontsize=12)
plt.ylabel('Stock / Issuer', fontsize=12)

for i, p in enumerate(barplot_hot.patches):
    width = p.get_width()
    plt.text(width + (width * 0.02), p.get_y() + p.get_height() / 2, f'+${width:,.1f}M', 
             ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize Coldest Stocks
plt.figure(figsize=(12, 8))
barplot_cold = sns.barplot(x='change_value_MMs', y='nameOfIssuer', data=coldest, palette='Reds_r')

plt.title(f'Coldest Stocks: Largest Drop ({pd.to_datetime(previous_period).strftime("%b %Y")} to {pd.to_datetime(current_period).strftime("%b %Y")})', fontsize=16, pad=15)
plt.xlabel('Change in Aggregated Value ($ MMs)', fontsize=12)
plt.ylabel('Stock / Issuer', fontsize=12)

# For negative values, we adjust text placement slightly differently
for i, p in enumerate(barplot_cold.patches):
    width = p.get_width()
    plt.text(width - (abs(width) * 0.02), p.get_y() + p.get_height() / 2, f'${width:,.1f}M', 
             ha='right', va='center', fontsize=10)

plt.tight_layout()
plt.show()